In [24]:
from ISLP import load_data
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, LassoCV, Lasso
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

In [2]:
Boston = load_data('Boston')
Boston_train, Boston_test = train_test_split(Boston, test_size=0.3, random_state=0)
Boston

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,6.48,22.0


In [3]:
preds = Boston.columns.drop('crim')
X_train = Boston_train[preds]
y_train = Boston_train['crim']
model = sm.OLS(y_train, X_train)
results = model.fit()


In [4]:
X_test= Boston_test[preds]
y_test = Boston_test['crim']
test_pred = results.predict(X_test)
mse = mean_squared_error(y_test, test_pred)
np.sqrt(mse)

6.719636122922913

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
ridge = Ridge()

In [7]:
param_grid = {'alpha': np.logspace(-4, 1, 100)}

In [8]:
grid_search = GridSearchCV(ridge, param_grid, cv=5, scoring='neg_mean_squared_error')

In [9]:
grid_search.fit(X_train_scaled, y_train)
best_alpha = grid_search.best_params_['alpha']
best_alpha

10.0

In [10]:
final_model = Ridge(alpha=best_alpha)
final_model.fit(X_train_scaled, y_train)

y_pred = final_model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
np.sqrt(mse)

6.682708241417807

In [11]:
lasso = Lasso()

In [12]:
grid_search = GridSearchCV(lasso,
                    param_grid,
                    scoring='neg_mean_squared_error',
                    cv=5)
grid_search.fit(X_train_scaled, y_train)
best_alpha = grid_search.best_params_['alpha']
best_alpha

0.03351602650938841

In [13]:
final_model = Lasso(alpha=best_alpha)
final_model.fit(X_train_scaled, y_train)

y_pred = final_model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
np.sqrt(mse)

6.6621480111273685

In [14]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('ridge', Ridge())
])

In [17]:
param_grid = {
    'pca__n_components': range(1, 13)
}

In [18]:
grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

,estimator,"Pipeline(step...e', Ridge())])"
,param_grid,"{'pca__n_components': range(1, 13)}"
,scoring,'neg_mean_squared_error'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [20]:
best_n_components = grid_search.best_params_['pca__n_components']
best_n_components

12

In [23]:
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
rmse

6.67634330173999

In [25]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pls', PLSRegression())
])
param_grid = {
    'pls__n_components': list(range(1, 13))
}


In [26]:
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

,estimator,Pipeline(step...egression())])
,param_grid,"{'pls__n_components': [1, 2, ...]}"
,scoring,'neg_mean_squared_error'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [28]:
best_n_components = grid.best_params_['pls__n_components']
best_n_components

10

In [29]:
y_pred = grid.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
rmse

6.676106150730389

In [ ]:
#Lasso worked the best with a RMSE of 6.662